# Notebook 51 — Analytic Approximation for b(Γ_eff)

Goal: derive an interpretable approximation for the stretching exponent b in terms of shape features of Γ_eff(x).


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit


## Generate example Γ_eff(x) families


In [ ]:
x = np.linspace(1e-3,1,400)

def gamma_family(x,a=2, slope=0.5, curv=0.5):
    return a + slope*x + curv*(x-0.5)**2

gammas = [gamma_family(x,2,s,c) for s,c in [(0.5,0.2),(1,0.5),(-0.5,0.8),(0.2,-0.3)]]


## Build S(x)


In [ ]:
def build_S(g):
    dx = x[1]-x[0]
    return np.exp(-np.cumsum(g)*dx)

Ss=[build_S(g) for g in gammas]


## Fit stretched exponential


In [ ]:
def stretched(x,a,b):
    return np.exp(-a*x**b)

def fit_b(S):
    popt,_=curve_fit(stretched,x,S,p0=[1,1],bounds=([0,0.1],[10,2]))
    return popt[1]

bs=[fit_b(S) for S in Ss]
bs


## Extract analytic features


In [ ]:
def features(g):
    d1=np.gradient(g,x)
    d2=np.gradient(d1,x)
    return np.mean(np.abs(d1)), np.mean(np.abs(d2)), np.std(g)/np.mean(g)

feats=[features(g) for g in gammas]
feats


## Fit simple analytic model: b ≈ α + β·slope + γ·curvature + δ·CV


In [ ]:
X=np.array(feats)
y=np.array(bs)

X_aug=np.column_stack([np.ones(len(X)),X])

coeffs=np.linalg.lstsq(X_aug,y,rcond=None)[0]
coeffs


## Prediction vs true


In [ ]:
y_pred=X_aug@coeffs

plt.scatter(y,y_pred)
plt.plot([min(y),max(y)],[min(y),max(y)],'--')
plt.xlabel("true b"); plt.ylabel("predicted b")
plt.title("Analytic approximation for b")
plt.show()


## Interpretation

We obtain an approximate closed-form:

b ≈ α + β·⟨|dΓ/dx|⟩ + γ·⟨|d²Γ/dx²|⟩ + δ·CV

This provides an interpretable bridge between dynamics and universality.
